In [ ]:
# Required imports
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Any, Union, Literal
from enum import Enum
from datetime import datetime
import json

# For validation (optional but recommended)
try:
    from pydantic import BaseModel, Field, ConfigDict
    USE_PYDANTIC = True
    print("✅ Using Pydantic for validation")
except ImportError:
    USE_PYDANTIC = False
    print("ℹ️ Using dataclasses (Pydantic not available)")

## 1. Message Roles and Types

In LLM conversations, messages have different **roles**:
- `system`: Instructions that guide the model's behavior
- `user`: Input from the human user
- `assistant`: Responses from the AI model
- `tool`: Results from tool/function execution

In [ ]:
# Define message roles as an Enum for type safety
class MessageRole(str, Enum):
    """Roles in a conversation."""
    SYSTEM = "system"      # Instructions for the model
    USER = "user"          # Human input
    ASSISTANT = "assistant"  # AI response
    TOOL = "tool"          # Tool execution result

# Example usage
print(f"Available roles: {[r.value for r in MessageRole]}")
print(f"Is 'user' a valid role? {MessageRole.USER.value == 'user'}")

In [ ]:
# Message class - represents a single message in a conversation

if USE_PYDANTIC:
    class Message(BaseModel):
        """A single message in a conversation."""
        model_config = ConfigDict(use_enum_values=True)
        
        role: MessageRole
        content: str
        name: Optional[str] = None  # For tool messages
        tool_call_id: Optional[str] = None  # Links to tool call
        metadata: Dict[str, Any] = Field(default_factory=dict)
        
        def to_dict(self) -> Dict[str, Any]:
            """Convert to API-compatible dictionary."""
            d = {"role": self.role, "content": self.content}
            if self.name:
                d["name"] = self.name
            if self.tool_call_id:
                d["tool_call_id"] = self.tool_call_id
            return d
else:
    @dataclass
    class Message:
        """A single message in a conversation."""
        role: MessageRole
        content: str
        name: Optional[str] = None
        tool_call_id: Optional[str] = None
        metadata: Dict[str, Any] = field(default_factory=dict)
        
        def to_dict(self) -> Dict[str, Any]:
            d = {"role": self.role.value, "content": self.content}
            if self.name:
                d["name"] = self.name
            if self.tool_call_id:
                d["tool_call_id"] = self.tool_call_id
            return d

print("✅ Message class defined")

In [ ]:
# Example: Creating messages for a conversation

# System message - sets up the AI's behavior
system_msg = Message(
    role=MessageRole.SYSTEM,
    content="You are a helpful math tutor. Show your reasoning step by step."
)

# User message - the question
user_msg = Message(
    role=MessageRole.USER,
    content="What is 15% of 80?"
)

# Assistant message - the AI's response
assistant_msg = Message(
    role=MessageRole.ASSISTANT,
    content="To find 15% of 80:\n1. Convert 15% to decimal: 15/100 = 0.15\n2. Multiply: 0.15 × 80 = 12\n\nAnswer: 15% of 80 is **12**"
)

# Build the conversation
conversation = [system_msg, user_msg, assistant_msg]

print("📝 Conversation:")
for msg in conversation:
    print(f"\n[{msg.role.value.upper()}]")
    print(msg.content)

## 2. Tool Results

When an AI agent uses tools (like calculators, search, etc.), we need to capture:
- **tool_name**: Which tool was used
- **tool_input**: What arguments were passed
- **output**: The result (can be any type)
- **success**: Whether execution succeeded
- **error**: Error message if failed

In [ ]:
if USE_PYDANTIC:
    class ToolResult(BaseModel):
        """Result from executing a tool."""
        model_config = ConfigDict(arbitrary_types_allowed=True)
        
        tool_name: str
        tool_input: Dict[str, Any]
        output: Any = None
        success: bool = True
        error: Optional[str] = None
        execution_time: Optional[float] = None  # In seconds
        
        @property
        def output_str(self) -> str:
            """String representation of output for LLM consumption."""
            if not self.success:
                return f"Error: {self.error}"
            if isinstance(self.output, str):
                return self.output
            return json.dumps(self.output, indent=2, default=str)
else:
    @dataclass
    class ToolResult:
        """Result from executing a tool."""
        tool_name: str
        tool_input: Dict[str, Any]
        output: Any = None
        success: bool = True
        error: Optional[str] = None
        execution_time: Optional[float] = None
        
        @property
        def output_str(self) -> str:
            if not self.success:
                return f"Error: {self.error}"
            if isinstance(self.output, str):
                return self.output
            return json.dumps(self.output, indent=2, default=str)

print("✅ ToolResult class defined")

In [ ]:
# Example: Successful tool execution

calc_result = ToolResult(
    tool_name="calculator",
    tool_input={"expression": "15 * 80 / 100"},
    output=12.0,
    success=True,
    execution_time=0.001
)

print(f"Tool: {calc_result.tool_name}")
print(f"Input: {calc_result.tool_input}")
print(f"Output: {calc_result.output_str}")
print(f"Success: {calc_result.success}")
print(f"Time: {calc_result.execution_time}s")

In [ ]:
# Example: Failed tool execution

failed_result = ToolResult(
    tool_name="web_search",
    tool_input={"query": "latest AI news"},
    output=None,
    success=False,
    error="Network timeout: Unable to reach search API"
)

print(f"Tool: {failed_result.tool_name}")
print(f"Output: {failed_result.output_str}")
print(f"Success: {failed_result.success}")

## 3. Reasoning Steps

For Chain-of-Thought (CoT) reasoning, we track each step:
- **step_number**: The sequence order
- **thought**: The reasoning content
- **action**: Optional action to take (for ReAct)
- **observation**: Result of the action
- **confidence**: How confident the model is (0-1)

In [ ]:
if USE_PYDANTIC:
    class ReasoningStep(BaseModel):
        """A single step in a reasoning chain."""
        model_config = ConfigDict(arbitrary_types_allowed=True)
        
        step_number: int
        thought: str
        action: Optional[str] = None  # Tool to call
        action_input: Optional[Dict[str, Any]] = None  # Tool args
        observation: Optional[str] = None  # Result of action
        confidence: Optional[float] = Field(default=None, ge=0.0, le=1.0)
        timestamp: datetime = Field(default_factory=datetime.now)
        
        def __str__(self) -> str:
            parts = [f"Step {self.step_number}: {self.thought}"]
            if self.action:
                parts.append(f"  Action: {self.action}({self.action_input})")
            if self.observation:
                parts.append(f"  Observation: {self.observation}")
            return "\n".join(parts)
else:
    @dataclass
    class ReasoningStep:
        """A single step in a reasoning chain."""
        step_number: int
        thought: str
        action: Optional[str] = None
        action_input: Optional[Dict[str, Any]] = None
        observation: Optional[str] = None
        confidence: Optional[float] = None
        timestamp: datetime = field(default_factory=datetime.now)
        
        def __str__(self) -> str:
            parts = [f"Step {self.step_number}: {self.thought}"]
            if self.action:
                parts.append(f"  Action: {self.action}({self.action_input})")
            if self.observation:
                parts.append(f"  Observation: {self.observation}")
            return "\n".join(parts)

print("✅ ReasoningStep class defined")

In [ ]:
# Example: Multi-step reasoning with ReAct pattern

steps = [
    ReasoningStep(
        step_number=1,
        thought="The user wants to know 15% of 80. I should use the calculator.",
        action="calculator",
        action_input={"expression": "0.15 * 80"},
        observation="12.0",
        confidence=0.95
    ),
    ReasoningStep(
        step_number=2,
        thought="The calculation shows 12. I can now provide the final answer.",
        confidence=0.99
    ),
]

print("🧠 Reasoning Chain:")
print("=" * 50)
for step in steps:
    print(step)
    print("-" * 50)

## 4. Agent State

For multi-step agents, we need to track:
- **messages**: Full conversation history
- **reasoning_steps**: Chain of reasoning
- **tool_results**: Results from tool calls
- **current_step**: Where we are in the process
- **status**: Current agent status

In [ ]:
class AgentStatus(str, Enum):
    """Status of an agent's execution."""
    IDLE = "idle"              # Not started
    THINKING = "thinking"      # Processing/reasoning
    ACTING = "acting"          # Executing a tool
    WAITING = "waiting"        # Waiting for input
    COMPLETED = "completed"    # Finished successfully
    ERROR = "error"            # Encountered an error

print(f"Agent statuses: {[s.value for s in AgentStatus]}")

In [ ]:
if USE_PYDANTIC:
    class AgentState(BaseModel):
        """Complete state of an agent during execution."""
        model_config = ConfigDict(arbitrary_types_allowed=True, use_enum_values=True)
        
        messages: List[Message] = Field(default_factory=list)
        reasoning_steps: List[ReasoningStep] = Field(default_factory=list)
        tool_results: List[ToolResult] = Field(default_factory=list)
        current_step: int = 0
        max_steps: int = 10
        status: AgentStatus = AgentStatus.IDLE
        error: Optional[str] = None
        metadata: Dict[str, Any] = Field(default_factory=dict)
        
        def add_message(self, role: MessageRole, content: str) -> None:
            """Add a message to the conversation."""
            self.messages.append(Message(role=role, content=content))
        
        def add_reasoning_step(self, thought: str, **kwargs) -> ReasoningStep:
            """Add a reasoning step."""
            step = ReasoningStep(
                step_number=len(self.reasoning_steps) + 1,
                thought=thought,
                **kwargs
            )
            self.reasoning_steps.append(step)
            return step
        
        @property
        def is_complete(self) -> bool:
            """Check if agent has finished."""
            return self.status in [AgentStatus.COMPLETED, AgentStatus.ERROR]
        
        @property
        def should_continue(self) -> bool:
            """Check if agent should continue processing."""
            return not self.is_complete and self.current_step < self.max_steps
else:
    @dataclass
    class AgentState:
        """Complete state of an agent during execution."""
        messages: List[Message] = field(default_factory=list)
        reasoning_steps: List[ReasoningStep] = field(default_factory=list)
        tool_results: List[ToolResult] = field(default_factory=list)
        current_step: int = 0
        max_steps: int = 10
        status: AgentStatus = AgentStatus.IDLE
        error: Optional[str] = None
        metadata: Dict[str, Any] = field(default_factory=dict)
        
        def add_message(self, role: MessageRole, content: str) -> None:
            self.messages.append(Message(role=role, content=content))
        
        def add_reasoning_step(self, thought: str, **kwargs) -> ReasoningStep:
            step = ReasoningStep(
                step_number=len(self.reasoning_steps) + 1,
                thought=thought,
                **kwargs
            )
            self.reasoning_steps.append(step)
            return step
        
        @property
        def is_complete(self) -> bool:
            return self.status in [AgentStatus.COMPLETED, AgentStatus.ERROR]
        
        @property
        def should_continue(self) -> bool:
            return not self.is_complete and self.current_step < self.max_steps

print("✅ AgentState class defined")

In [ ]:
# Example: Simulating agent execution

state = AgentState(max_steps=5)

# Initialize
state.add_message(MessageRole.SYSTEM, "You are a helpful assistant with access to a calculator.")
state.add_message(MessageRole.USER, "What is 15% of 80?")
state.status = AgentStatus.THINKING

# Step 1: Reasoning
state.add_reasoning_step(
    thought="I need to calculate 15% of 80. I'll use the calculator.",
    action="calculator",
    action_input={"expression": "0.15 * 80"}
)
state.current_step += 1
state.status = AgentStatus.ACTING

# Execute tool
result = ToolResult(
    tool_name="calculator",
    tool_input={"expression": "0.15 * 80"},
    output=12.0,
    success=True
)
state.tool_results.append(result)
state.reasoning_steps[-1].observation = str(result.output)

# Step 2: Final answer
state.add_reasoning_step(
    thought="The calculator returned 12.0. This is the answer.",
    confidence=0.99
)
state.current_step += 1
state.add_message(MessageRole.ASSISTANT, "15% of 80 equals **12**.")
state.status = AgentStatus.COMPLETED

# Print state summary
print(f"📊 Agent State Summary")
print(f"  Status: {state.status.value}")
print(f"  Steps: {state.current_step}/{state.max_steps}")
print(f"  Messages: {len(state.messages)}")
print(f"  Reasoning Steps: {len(state.reasoning_steps)}")
print(f"  Tool Calls: {len(state.tool_results)}")
print(f"  Complete: {state.is_complete}")

## 5. LLM Response Types

When calling LLM APIs, we get structured responses containing:
- **content**: The generated text
- **tool_calls**: Any function/tool calls requested
- **usage**: Token counts
- **model**: Which model was used

In [ ]:
if USE_PYDANTIC:
    class ToolCall(BaseModel):
        """A tool call requested by the LLM."""
        id: str
        name: str
        arguments: Dict[str, Any]
    
    class TokenUsage(BaseModel):
        """Token usage statistics."""
        prompt_tokens: int = 0
        completion_tokens: int = 0
        total_tokens: int = 0
    
    class LLMResponse(BaseModel):
        """Response from an LLM API call."""
        model_config = ConfigDict(arbitrary_types_allowed=True)
        
        content: Optional[str] = None
        tool_calls: List[ToolCall] = Field(default_factory=list)
        usage: TokenUsage = Field(default_factory=TokenUsage)
        model: str = ""
        finish_reason: Optional[str] = None
        raw_response: Optional[Any] = None
        
        @property
        def has_tool_calls(self) -> bool:
            """Check if response includes tool calls."""
            return len(self.tool_calls) > 0
else:
    @dataclass
    class ToolCall:
        id: str
        name: str
        arguments: Dict[str, Any]
    
    @dataclass
    class TokenUsage:
        prompt_tokens: int = 0
        completion_tokens: int = 0
        total_tokens: int = 0
    
    @dataclass
    class LLMResponse:
        content: Optional[str] = None
        tool_calls: List[ToolCall] = field(default_factory=list)
        usage: TokenUsage = field(default_factory=TokenUsage)
        model: str = ""
        finish_reason: Optional[str] = None
        raw_response: Any = None
        
        @property
        def has_tool_calls(self) -> bool:
            return len(self.tool_calls) > 0

print("✅ LLMResponse classes defined")

In [ ]:
# Example: Response with tool call

response_with_tool = LLMResponse(
    content=None,  # No text content when making tool call
    tool_calls=[
        ToolCall(
            id="call_abc123",
            name="calculator",
            arguments={"expression": "0.15 * 80"}
        )
    ],
    usage=TokenUsage(prompt_tokens=50, completion_tokens=20, total_tokens=70),
    model="gpt-4",
    finish_reason="tool_calls"
)

print(f"Response has tool calls: {response_with_tool.has_tool_calls}")
print(f"Tool: {response_with_tool.tool_calls[0].name}")
print(f"Args: {response_with_tool.tool_calls[0].arguments}")
print(f"Tokens used: {response_with_tool.usage.total_tokens}")

In [ ]:
# Example: Regular text response

text_response = LLMResponse(
    content="15% of 80 equals 12. Here's how I calculated it:\n1. Convert 15% to 0.15\n2. Multiply: 0.15 × 80 = 12",
    tool_calls=[],
    usage=TokenUsage(prompt_tokens=45, completion_tokens=35, total_tokens=80),
    model="gpt-4",
    finish_reason="stop"
)

print(f"Response has tool calls: {text_response.has_tool_calls}")
print(f"Content: {text_response.content}")
print(f"Finish reason: {text_response.finish_reason}")

## 6. Putting It All Together

Let's create a complete example showing how all these types work together:

In [ ]:
def simulate_agent_interaction(query: str) -> AgentState:
    """
    Simulate a complete agent interaction.
    This shows how all the types work together.
    """
    # Initialize state
    state = AgentState(max_steps=5)
    state.add_message(MessageRole.SYSTEM, "You are a math assistant with a calculator.")
    state.add_message(MessageRole.USER, query)
    
    print(f"\n{'='*60}")
    print(f"🤖 Agent Processing: {query}")
    print(f"{'='*60}")
    
    # Step 1: Think and decide to use tool
    state.status = AgentStatus.THINKING
    state.add_reasoning_step(
        thought="I need to calculate this. Let me use the calculator.",
        action="calculator",
        action_input={"expression": "0.15 * 80"}
    )
    state.current_step += 1
    print(f"\n📝 {state.reasoning_steps[-1]}")
    
    # Step 2: Execute tool
    state.status = AgentStatus.ACTING
    result = ToolResult(
        tool_name="calculator",
        tool_input={"expression": "0.15 * 80"},
        output=12.0,
        success=True,
        execution_time=0.001
    )
    state.tool_results.append(result)
    state.reasoning_steps[-1].observation = result.output_str
    print(f"\n🔧 Tool Result: {result.output_str}")
    
    # Step 3: Formulate response
    state.status = AgentStatus.THINKING
    state.add_reasoning_step(
        thought=f"The calculator returned {result.output}. I can now give the final answer.",
        confidence=0.99
    )
    state.current_step += 1
    print(f"\n📝 {state.reasoning_steps[-1]}")
    
    # Complete
    final_answer = f"15% of 80 is **{result.output}**."
    state.add_message(MessageRole.ASSISTANT, final_answer)
    state.status = AgentStatus.COMPLETED
    
    print(f"\n{'='*60}")
    print(f"✅ Final Answer: {final_answer}")
    print(f"{'='*60}")
    
    return state

# Run the simulation
final_state = simulate_agent_interaction("What is 15% of 80?")

In [ ]:
# View complete state
print("\n📊 COMPLETE AGENT STATE")
print("=" * 60)

print(f"\n📬 Messages ({len(final_state.messages)}):")
for i, msg in enumerate(final_state.messages, 1):
    print(f"  {i}. [{msg.role.value}] {msg.content[:50]}..." if len(msg.content) > 50 else f"  {i}. [{msg.role.value}] {msg.content}")

print(f"\n🧠 Reasoning Steps ({len(final_state.reasoning_steps)}):")
for step in final_state.reasoning_steps:
    print(f"  Step {step.step_number}: {step.thought[:50]}..." if len(step.thought) > 50 else f"  Step {step.step_number}: {step.thought}")

print(f"\n🔧 Tool Results ({len(final_state.tool_results)}):")
for result in final_state.tool_results:
    print(f"  - {result.tool_name}: {result.output_str}")

print(f"\n📈 Status: {final_state.status.value}")
print(f"   Steps: {final_state.current_step}/{final_state.max_steps}")
print(f"   Complete: {final_state.is_complete}")

## Summary

In this notebook, we covered the **core types** used in AI reasoning systems:

| Type | Purpose |
|------|--------|
| `MessageRole` | Enum defining conversation roles (system, user, assistant, tool) |
| `Message` | Single message in a conversation with role and content |
| `ToolResult` | Captures tool execution output, success/failure, timing |
| `ReasoningStep` | Tracks thought, action, observation in CoT/ReAct |
| `AgentStatus` | Current state of agent (idle, thinking, acting, completed, error) |
| `AgentState` | Complete agent state with messages, steps, results |
| `ToolCall` | LLM's request to call a specific tool |
| `TokenUsage` | Token consumption statistics |
| `LLMResponse` | Structured response from LLM API |

These types form the **foundation** for building AI agents and reasoning systems!